# VUSA AI Quant - research walkthrough

Uses the same backend as the GUI. Set `SYNTHETIC = False` on a machine with internet access to use live data.

**Decision support only - not investment advice.**

In [ ]:
import sys, warnings; sys.path.insert(0, '..'); sys.path.insert(0, '../..'); warnings.filterwarnings('ignore')
from vusa_quant_ai.config.settings import load_settings
from vusa_quant_ai.services.api import Backend
SYNTHETIC = True
s = load_settings(); b = Backend(s)
res = b.analyze('fast', synthetic=SYNTHETIC or None, progress=lambda m, f: print(m))
res.ok, res.state['decision']['label']

## Decision components and the counter-case

In [ ]:
import pandas as pd
d = res.state['decision']
pd.DataFrame(d['components']).T[['score', 'coverage', 'evidence']]

In [ ]:
res.state['committee']['skeptic']

## Out-of-sample model comparison (primary horizon)

In [ ]:
fp = res.state['forecast_primary']
pd.DataFrame(fp['model_scores']).T.sort_values('ic_spearman', ascending=False)

In [ ]:
pd.DataFrame(fp['ensemble_comparison']).T

## Ablation: does each data group actually help?

In [ ]:
b.ablation(20)

## Walk-forward backtest (leakage audit runs first)

In [ ]:
bt = b.backtest(20, retrain_every=126)
pd.DataFrame({'strategy': bt.result.metrics, 'buy_and_hold': bt.result.benchmark_metrics})